# Bandit Algorithms
**Reinforcement Learning Course** — Paul-Antoine LE TOLGUENEC

---

**Problem.** At each round $t = 1,\ldots,T$, an agent pulls arm $A_t \in \mathcal{A} = \{1,\ldots,K\}$ and observes $R_t \sim \nu_{A_t}$. The goal is to minimise cumulative regret:
$$\text{Reg}_T = T\mu^* - \mathbb{E}\!\left[\sum_{t=1}^T R_t\right]$$

In [ ]:
!pip install -q plotly numpy

In [ ]:
import numpy as np
import plotly.graph_objects as go

## 1 · Bandit Environment

In [ ]:
class BernoulliBandit:
    """K-armed Bernoulli bandit. mu[a] = P(R=1 | A=a)."""
    def __init__(self, mu: np.ndarray):
        self.mu      = np.asarray(mu)
        self.K       = len(mu)
        self.mu_star = mu.max()

    def pull(self, a: int) -> float:
        return float(np.random.random() < self.mu[a])

    def regret(self, rewards: np.ndarray) -> np.ndarray:
        T = len(rewards)
        return np.arange(1, T + 1) * self.mu_star - np.cumsum(rewards)

## 2 · ε-Greedy

**Init:** $\hat{\mu}_a \leftarrow 0,\; N_a \leftarrow 0 \quad \forall a \in \mathcal{A}$  
**For** $t = 1,\ldots,T$: $B_t \sim \text{Bernoulli}(\varepsilon)$, then
$A_t \sim \text{Uniform}(\mathcal{A})$ if $B_t=1$ else $A_t = \arg\max_{a}\hat{\mu}_a$

> **TODO** — complete the `select` method (1 line).

In [ ]:
class EpsilonGreedy:
    def __init__(self, K: int, epsilon: float):
        self.K, self.epsilon = K, epsilon
        self.mu_hat = np.zeros(K)
        self.N      = np.zeros(K)

    def select(self) -> int:
        if np.random.random() < self.epsilon:
            return np.random.randint(self.K)
        return -1 #--TODO: return the greedy arm (1 line)

    def update(self, a: int, r: float):
        self.N[a] += 1
        self.mu_hat[a] += (r - self.mu_hat[a]) / self.N[a]

## 3 · Upper Confidence Bound

**For** $t = 1,\ldots,T$:
$A_t = \arg\max_{a \in \mathcal{A}}\!\left(\hat{\mu}_a + c\sqrt{\dfrac{\ln t}{N_a}}\right)$

In [ ]:
class UCB:
    def __init__(self, K: int, c: float = 2.0):
        self.K, self.c = K, c
        self.mu_hat = np.zeros(K)
        self.N      = np.zeros(K)
        self.t      = 0

    def select(self) -> int:
        self.t += 1
        if self.t <= self.K:          # pull each arm once first
            return self.t - 1
        return int(np.argmax(self.mu_hat + self.c * np.sqrt(np.log(self.t) / self.N)))

    def update(self, a: int, r: float):
        self.N[a] += 1
        self.mu_hat[a] += (r - self.mu_hat[a]) / self.N[a]

## 4 · Thompson Sampling

**Init:** $\alpha_a \leftarrow \alpha_0,\; \beta_a \leftarrow \beta_0 \quad \forall a \in \mathcal{A}$  
**For** $t = 1,\ldots,T$: $\theta_a \sim \text{Beta}(\alpha_a, \beta_a) \; \forall a$, then $A_t = \arg\max_a \theta_a$

In [ ]:
class ThompsonSampling:
    def __init__(self, K: int, alpha0: float = 1.0, beta0: float = 1.0):
        self.K     = K
        self.alpha = np.full(K, alpha0, dtype=float)
        self.beta  = np.full(K, beta0,  dtype=float)

    def select(self) -> int:
        return int(np.argmax(np.random.beta(self.alpha, self.beta)))

    def update(self, a: int, r: float):
        self.alpha[a] += r
        self.beta[a]  += 1 - r

## 5 · HP Sensitivity

Fixed bandit instance, `N_RUNS` independent runs per configuration. Shaded area = ±1 std error.

In [ ]:
MU     = np.array([0.2, 0.5, 0.75, 0.6, 0.4])
T      = 1000
N_RUNS = 200

COLORS = {"eg": "#a78bfa", "ucb": "#00d4ff", "ts": "#f59e0b"}

def hex_rgba(h: str, a: float) -> str:
    h = h.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{a})"

def run(algo_fn, T: int = T, N: int = N_RUNS) -> np.ndarray:
    """Returns (N, T) cumulative regret matrix."""
    env = BernoulliBandit(MU)
    out = np.zeros((N, T))
    for i in range(N):
        algo = algo_fn()
        rs = []
        for _ in range(T):
            a = algo.select()
            r = env.pull(a)
            algo.update(a, r)
            rs.append(r)
        out[i] = env.regret(np.array(rs))
    return out

def add_band(fig: go.Figure, mat: np.ndarray, color: str, name: str):
    mean = mat.mean(0)
    se   = mat.std(0) / np.sqrt(len(mat))
    x    = list(range(1, T + 1))
    fig.add_trace(go.Scatter(
        x=x + x[::-1], y=list(mean + se) + list((mean - se)[::-1]),
        fill="toself", fillcolor=hex_rgba(color, 0.15),
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ))
    fig.add_trace(go.Scatter(
        x=x, y=mean, name=name, line=dict(color=color, width=2),
    ))

In [ ]:
# ── ε-Greedy ──────────────────────────────────────────────────────────────────
epsilons  = [0.01, 0.05, 0.1, 0.3]
shades_eg = ["#ede9fe", "#c4b5fd", "#a78bfa", "#7c3aed"]

fig_eg = go.Figure()
for eps, col in zip(epsilons, shades_eg):
    add_band(fig_eg, run(lambda e=eps: EpsilonGreedy(len(MU), e)), col, f"ε = {eps}")

fig_eg.update_layout(
    title="ε-Greedy — HP sensitivity", xaxis_title="t", yaxis_title="Regret",
    template="plotly_dark", width=800, height=400, legend=dict(orientation="h", y=1.12),
)
fig_eg.show()

In [ ]:
# ── UCB ───────────────────────────────────────────────────────────────────────
cs         = [0.5, 1.0, 2.0, 4.0]
shades_ucb = ["#cffafe", "#67e8f9", "#00d4ff", "#0e7490"]

fig_ucb = go.Figure()
for c, col in zip(cs, shades_ucb):
    add_band(fig_ucb, run(lambda c_=c: UCB(len(MU), c_)), col, f"c = {c}")

fig_ucb.update_layout(
    title="UCB — HP sensitivity", xaxis_title="t", yaxis_title="Regret",
    template="plotly_dark", width=800, height=400, legend=dict(orientation="h", y=1.12),
)
fig_ucb.show()

In [ ]:
# ── Thompson Sampling ─────────────────────────────────────────────────────────
priors    = [(1, 1), (1, 5), (5, 1), (5, 5)]
shades_ts = ["#fef3c7", "#fcd34d", "#f59e0b", "#b45309"]

fig_ts = go.Figure()
for (a0, b0), col in zip(priors, shades_ts):
    add_band(fig_ts, run(lambda a=a0, b=b0: ThompsonSampling(len(MU), a, b)), col, f"α₀={a0}, β₀={b0}")

fig_ts.update_layout(
    title="Thompson Sampling — prior sensitivity", xaxis_title="t", yaxis_title="Regret",
    template="plotly_dark", width=800, height=400, legend=dict(orientation="h", y=1.12),
)
fig_ts.show()

## 6 · Best-of Comparison

Each algorithm is run with its best hyperparameter identified above.

In [ ]:
best = [
    ("ε-Greedy",          lambda: EpsilonGreedy(len(MU), 0.01),    COLORS["eg"]),
    ("UCB",               lambda: UCB(len(MU), 0.5),               COLORS["ucb"]),
    ("Thompson Sampling", lambda: ThompsonSampling(len(MU), 5, 5 ), COLORS["ts"]),
]

fig = go.Figure()
for name, algo_fn, color in best:
    add_band(fig, run(algo_fn), color, name)

fig.update_layout(
    title="Algorithm comparison — best HP",
    xaxis_title="t", yaxis_title="Cumulative regret",
    template="plotly_dark", width=900, height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()